# Scrape Google Trends for Your Own Topic - Lane B (backup: the agent's output, captured)

**SISMID 2026 - Day 1, 3:30 session.** Point a reliable workflow at **your own** disease
and place.

> **We use the Google Trends API, not scraping.** `pytrends` scrapes an unofficial
> endpoint: it hands out HTTP 429s and re-samples its numbers on every pull. The Trends
> API (`getGraph`) returns the **same normalized 0-100 index**, from a documented API,
> with a key. Same signal, no roulette. You get a key in class.

> Each cell is a captured example of what a coding agent (Codex, Claude Code, or
> Antigravity CLI) produces from the **Lane A** prompts.


## Step 0: the helper, your topic, and your key

Unlock the class secrets **in the terminal**, then start (or restart) Jupyter so the
kernel inherits them:

```bash
source scripts/unlock-gt-api-key.sh     # asks for the Google Trends passcode
```

That decrypts `secrets/gt-api/GT_API.enc` and exports **`GT_API`** (it also saves it to
`~/.bashrc`, so new terminals stay unlocked). Never paste a key into a notebook.
If no key is set, everything below still runs from the cached snapshot.


In [1]:
# --- Produced by the agent from the Plan A / Step 0 prompt ---
import pandas as pd, matplotlib.pyplot as plt, os, time, json
import urllib.request, urllib.parse, urllib.error

# ===== EDIT THESE TWO LINES for your own topic =====
MY_TERMS = ['influenza', 'flu', 'fever']   # terms, or topic mids like '/m/0cycc'
MY_GEO   = 'US'                            # 'US', 'MX', 'US-GA', 'BR' (ISO-3166)
# ===================================================
# Dates are YYYY-MM. Google picks the resolution from the window, exactly as the
# Trends website does: ~18 months or less gives weekly points, longer gives monthly.
START, END = '2021-07', time.strftime('%Y-%m')

# The class API key. Set it in the terminal before starting Jupyter:
# (that decrypts secrets/GT_API.enc with the class passcode)
# NEVER paste a key into a notebook you will commit or share.
API_KEY = os.getenv('GT_API', '')
print('API key loaded' if API_KEY else 'No GT_API set - the cached snapshot will be used')

ENDPOINT = 'https://www.googleapis.com/trends/v1beta/graph'
CACHE_PATHS = [
    '../data/google_trends_flu_us_cached.csv',
    'data/google_trends_flu_us_cached.csv',
    './google_trends_flu_us_cached.csv',
]

def _norm(c):
    return c.strip().replace(' ', '_').replace('/', '_').lstrip('_')

def gt_api_fetch(terms, geo=None, start=None, end=None, key=None):
    """Google Trends API (getGraph).

    Returns the SAME normalized 0-100 index pytrends returns, but from a
    documented Google API: no HTTP 429 roulette, and identical values on every
    pull. Tidy output: a date column plus one column per term.
    """
    geo, start, end = geo or MY_GEO, start or START, end or END
    key = key or API_KEY
    if not key:
        raise RuntimeError("GT_API not set (export GT_API='...')")
    params = [('key', key)] + [('terms', t) for t in terms] + [
        ('restrictions.geo', geo),
        ('restrictions.startDate', start),
        ('restrictions.endDate', end)]
    url = ENDPOINT + '?' + urllib.parse.urlencode(params)
    try:
        with urllib.request.urlopen(url, timeout=45) as r:
            lines = json.loads(r.read()).get('lines') or []
    except urllib.error.HTTPError as e:
        raise RuntimeError(f'HTTP {e.code}: {e.read().decode("utf-8","replace")[:200]}') from e
    if not lines:
        raise RuntimeError(f'empty response for {terms} in {geo}')
    frames = [pd.DataFrame({
                 'date': pd.to_datetime([p['date'] for p in l['points']]),
                 _norm(l['term']): [p['value'] for p in l['points']]}).set_index('date')
              for l in lines]
    return pd.concat(frames, axis=1).reset_index().sort_values('date').reset_index(drop=True)

def load_cache():
    for p in CACHE_PATHS:
        if os.path.exists(p):
            print('Using cached example (flu, US):', p)
            return pd.read_csv(p, parse_dates=['date'])
    raise FileNotFoundError('Cached example not found; check the data/ folder.')

def gt_fetch(terms, geo=None, start=None, end=None):
    """API first, cached snapshot if the key is missing or the call fails."""
    try:
        df = gt_api_fetch(terms, geo, start, end)
        gap = df['date'].diff().dt.days.median()
        print(f"Trends API: {len(df)} points "
              f"({'weekly' if gap <= 10 else 'monthly'}), "
              f"{df['date'].min().date()} -> {df['date'].max().date()}")
        return df
    except Exception as e:
        print('Trends API unavailable:', e)
        return load_cache()


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/vscode/.local/lib/python3.12/site-packages/traitlets/traitlets.py", line 651, in get
    value = obj._trait_values[self.name]
            ~~~~~~~~~~~~~~~~~^^^^^^^^^^^
KeyError: '_control_lock'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/vscode/.local/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/home/vscode/.local/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 347, in dispatch_control
    async with self._control_lock:
               ^^^^^^^^^^^^^^^^^^
  File "/home/vscode/.local/lib/python3.12/site-packages/traitlets/traitlets.py", line 706, in __get__
    return self.get(obj, cls)  # type:ignore[return-value]
           ^^^^^^^^^^^^^^^^^^
  File "/home/vscode/.local/lib/python3.12/site-packages/traitlets/traitlets.py", line 668,

## Step 1: pull your topic

One call. No retry loop, no rate-limit dance.


In [ ]:
# --- Produced by the agent from the Plan A / Step 1 prompt ---
series = gt_fetch(MY_TERMS)
cols = [c for c in series.columns if c != 'date']
plt.figure(figsize=(10,4))
for c in cols:
    plt.plot(series['date'], series[c], label=c)
plt.legend(); plt.title(f'Google Trends ({MY_GEO}): normalized search interest')
plt.ylabel('interest (0-100)'); plt.xlabel('date'); plt.tight_layout(); plt.show()
series.tail()


## Step 2: term vs topic (the language lesson)

A **term** is the literal string you typed. A **topic** is Google's Knowledge Graph entity
(a `/m/...` id) that folds together every spelling, synonym and **language** for the same
concept. The API takes either, so you can compare them in one call.

This matters most **outside English-speaking countries**: the English term `flu` is nearly
silent in France, Italy or Mexico, while the Influenza topic `/m/0cycc` picks up *grippe*,
*influenza* and *gripe* alike.

Useful flu topic ids (from our own pipeline): `/m/0cycc` Influenza, `/m/0416v7` Influenza
Vaccine, `/m/0cjf0` Fever, `/m/01b_21` Cough, `/m/0n073` Common Cold.


In [ ]:
# --- Produced by the agent from the Plan A / Step 2 prompt ---
try:
    rows = []
    for geo, label in [('US','United States'), ('FR','France'),
                       ('IT','Italy'), ('MX','Mexico')]:
        c = gt_api_fetch(['flu', '/m/0cycc'], geo=geo, start='2022-01')
        c.columns = ['date', 'term_flu', 'topic_influenza']
        rows.append({'geo': label,
                     'term "flu" max': int(c['term_flu'].max()),
                     'topic max': int(c['topic_influenza'].max()),
                     'term mean': round(c['term_flu'].mean(), 1),
                     'topic mean': round(c['topic_influenza'].mean(), 1)})
        if geo == 'FR': fr = c
    display(pd.DataFrame(rows))
    plt.figure(figsize=(10,4))
    plt.plot(fr['date'], fr['term_flu'], label='term: "flu" (English)')
    plt.plot(fr['date'], fr['topic_influenza'], label='topic: /m/0cycc (Influenza)')
    plt.legend(); plt.title('France: an English term sees almost nothing; the topic sees the season')
    plt.ylabel('interest (0-100)'); plt.tight_layout(); plt.show()
except Exception as e:
    print('Needs a key for the live comparison:', e)


## Step 3: why this is the reliable path

The headline difference from `pytrends`: **the API is deterministic.** Pull the same
series twice and you get the same numbers, so your analysis is reproducible.

`pytrends` reads a sampled public endpoint, so repeat pulls disagree, and it rate-limits
you. That sampling noise is real and it is why the afternoon session is about statistical
preprocessing. It is a property of the *scraped* endpoint, not of Google Trends itself.


In [ ]:
try:
    a = gt_api_fetch(MY_TERMS[:1]); b = gt_api_fetch(MY_TERMS[:1])
    col = [c for c in a.columns if c != 'date'][0]
    same = a[col].tolist() == b[col].tolist()
    print('two API pulls identical?', same)
except Exception as e:
    print('Needs a key:', e)


## Step 4: sanity-check before you trust it


In [ ]:
# --- Produced by the agent from the Plan A / Step 4 prompt ---
print('geo        :', MY_GEO)
print('date range :', series['date'].min().date(), 'to', series['date'].max().date())
print('n rows     :', len(series))
gap = series['date'].diff().dt.days.median()
print('resolution :', 'weekly' if gap <= 10 else 'monthly', f'({gap:.0f} day spacing)')
print('missing per column:'); print(series[cols].isna().sum())
print('% zeros per column:'); print((series[cols] == 0).mean().mul(100).round(1))


## Step 5: save your tidy CSV


In [ ]:
series.to_csv('my_topic_search.csv', index=False)
print('saved my_topic_search.csv with', len(series), 'rows and columns', list(series.columns))


## Reflection

- Same signal as `pytrends` (normalized 0-100), delivered reliably: no 429s, and
  **identical numbers on every pull**, which makes your work reproducible.
- The API takes **terms or topic ids**, so the term-vs-topic comparison is one call.
- Window length chooses the resolution, exactly as on the Trends website: about 18
  months or less gives weekly points, longer gives monthly.
- **Quota and etiquette:** 1,000 requests/day on the class key, and health-related
  topics only. Do not commit the key.

**Stretch:** swap `MY_TERMS` and `MY_GEO` for a teammate's disease and place, or try a
region code such as `US-GA`.
